# 4. 의사결정나무와 랜덤 포레스트 회귀

회귀 트리는 자식 노드의 가중 제곱 오차가 낮아지도록 분할합니다. 랜덤 포레스트는 여러 트리의 예측을 평균합니다. 제공된 소규모 CSV의 점수는 분할에 민감합니다.

원본 노트북 Lv2_Python_1118_DecisionTree_regression_house.ipynb

**Decision Tree(의사결정나무) - 회귀**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = DATA_DIR if 'DATA_DIR' in globals() else Path('data')
if not (data_dir / 'house.price.csv').is_file():
    data_dir = Path('Study_Python') / '[7] Machine_Learning' / 'data'
if not (data_dir / 'house.price.csv').is_file():
    data_dir = Path('..') / 'Study_Python' / '[7] Machine_Learning' / 'data'
file_path = data_dir / 'house.price.csv'
df = pd.read_csv(file_path)

In [ ]:
df.head()
df.count()

In [ ]:
# 불필요한 열 제거: ID
df = df.drop(columns=["ID"])

In [ ]:
# 입력(X), 타겟(y) 설정
X = df[["year", "size"]].values     # 입력 변수
y = df["price"].values             # 예측 대상

In [ ]:
# 데이터셋을 dataframe 형태로 변환
X_df = pd.DataFrame(X, columns=["year", "size"])
y_df = pd.DataFrame(y, columns=["price"])

X_df.head()
y_df.head()

In [ ]:
# 데이터를 훈련 세트와 테스트 세트로 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# y_train.mean()

In [ ]:
# Random Forest 회귀 모델 생성
rf_reg = RandomForestRegressor(
    n_estimators=200,      # 트리 개수(적당히 크게)
    max_depth=3,
    random_state=1
    )

In [ ]:
# 모델 학습
rf_reg.fit(X_train, y_train)

In [ ]:
# 테스트 데이터에 대한 예측 수행
y_pred = rf_reg.predict(X_test)

# 성능 지표 계산 및 출력
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))
print("Coefficient of Determination (R^2):", r2_score(y_test, y_pred))

In [ ]:
# 변수 중요도 확인
print("\nFeature Importances:")
for name, imp in zip(["year", "size"], rf_reg.feature_importances_):
    print(f"- {name}: {imp:.4f}")

In [ ]:
### 회귀 트리(RandomForestRegressor) 노드 분할
#  "비슷한 숫자(target 값)끼리 한 그룹에 모이도록 선을 긋는 과정

# 분류 모델이 ex, "사과와 배를 잘 구분하는 기준(rule)"을 찾기
# 회귀 모델 "노드들을 잘 나눴을 때 각 그룹 내부의 값들이 서로 비슷한(분산이나 오차가 작아지도록) 만드는 분리 하기

# ex 학생 4명의 공부시간, 점수 데이터
# 분할 기준 squared_error (평균 제곱 오차, MSE) 사용
# 학생 A: 공부 1시간 , 50점
# 학생 B: 공부 2시간 , 60점
# 학생 C: 공부 5시간 , 90점
# 학생 D: 공부 6시간 , 100점


# step 1: 분할 전 상태 - 부모 노드
# 현재 4명의 데이터가 모두 한 노드에(부모 노드)..
   # 이 노드의 예측값 (평균): (50 + 60 + 90 + 100) ÷ 4 = 75점
   # 이 노드의 오차 (MSE): 각 점수와 평균(75점)의 차이를 제곱해 평균 : 425

# step 2: 가능한 모든 분할 기준(임계값) 후보 테스트
# 공부 시간을 기준으로 오차가 얼마나 줄어드는지 계산
  # 후보 1: '공부 시간 ≤ 1.5시간' 으로 구분
	   # 왼쪽 노드 (1명: 50점): 평균 = 50점 / 오차(MSE) = 0
	   # 오른쪽 노드 (3명: 60, 90, 100점): 평균 = 83.3점 / 오차(MSE) ≈ 288.9
	   # 분할 후 전체 가중 평균 오차: (1/4 × 0) + (3/4 × 288.9) ≈ 216.7
   # 후보 2: '공부 시간 ≤ 3.5시간'으로 구분
       # 왼쪽 노드 (2명: 50, 60점) ....
       # 오른쪽 노드 (2명: 90, 100점) ...
	   # 분할 후 전체 가중 평균 오차: (2/4 × 25) + (2/4 × 25) = 25
   # 후보 3: '공부 시간 ≤ 5.5시간'으로  구분... 216.7

# stpe 3: 최적의 분할 기준 선택
# 위 셋 중 분할 후 전체 오차가 가장 작아지는(425 -> 25) 후보 2('공부 시간 ≤ 3.5시간')를 분할 기준으로 선택

# 이와 같이 rule 세팅 후 추론....
# 노드 분할 기존에서 왼쪽으로 분류된 학생 (3.5시간 이하) -> 55점으로 예측 (2명: 50, 60점)
# 노드 분할 기존에서 오른으로 분류된 학생 (3.5시간 초과) -> 95점으로 예측 (2명: 90, 100점)

#### DT - 회귀
# 한 노드에 들어간 숫자들의 평균값과 실제값의 차이(오차)가 최소가 되도록

In [ ]:
### house 데이터

# 부모노드 (전체 데이터)
# 전체 평균 가격(예측값): 3.85
# 현재 오차(MSE): 1.1125

# 노드 분할
# ex,  size <= 50.0
# ex,  year <= 8.5

# 최적의 분할 기준 탐색